# Backpropagation: complete recap

This notebook is a full recap of **backpropagation** from intuition to math to a worked numerical example to code.

Backpropagation is the algorithm that tells a neural network **how each weight and bias contributed to the final error**, so the network can adjust them to reduce that error.

At a high level, training a neural network does this repeatedly:

1. **Forward pass**: compute the prediction from the input.
2. **Loss computation**: measure how wrong the prediction is.
3. **Backward pass (backpropagation)**: compute gradients of the loss with respect to every parameter.
4. **Parameter update**: move parameters a little in the direction that reduces the loss.

The heart of backpropagation is just the **chain rule from calculus** applied systematically.


## 1. The core idea

Suppose the loss is $L$, and a weight is $w$.

We want:

$$
\frac{\partial L}{\partial w}
$$

This tells us:

- if positive: increasing $w$ increases loss, so decrease $w$
- if negative: increasing $w$ decreases loss, so increase $w$
- magnitude tells how sensitive the loss is to that weight

Then gradient descent updates:

$$
w \leftarrow w - \eta \frac{\partial L}{\partial w}
$$

where $\eta$ is the learning rate.


## 2. Why the chain rule appears

In a neural network, a weight usually does **not** affect the loss directly. It affects some intermediate quantity, which affects another quantity, and eventually the output and the loss.

So if:

$$
w \to z \to a \to L
$$

then by the chain rule:

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial a}
\cdot
\frac{\partial a}{\partial z}
\cdot
\frac{\partial z}{\partial w}
$$

That is all backpropagation really is: **propagating gradients backward through the computational graph.**


## 3. Single neuron first

Let one neuron compute:

$$
z = wx + b
$$

$$
a = f(z)
$$

Suppose the target is $y$, and we use mean squared error for one example:

$$
L = \frac{1}{2}(a-y)^2
$$

We want gradients with respect to $w$ and $b$.

Using chain rule:

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial a}
\cdot
\frac{\partial a}{\partial z}
\cdot
\frac{\partial z}{\partial w}
$$

Now compute each piece:

$$
\frac{\partial L}{\partial a} = a-y
$$

$$
\frac{\partial a}{\partial z} = f'(z)
$$

$$
\frac{\partial z}{\partial w} = x
$$

So:

$$
\frac{\partial L}{\partial w} = (a-y)f'(z)x
$$

Similarly:

$$
\frac{\partial L}{\partial b}
=
\frac{\partial L}{\partial a}
\cdot
\frac{\partial a}{\partial z}
\cdot
\frac{\partial z}{\partial b}
=
(a-y)f'(z)\cdot 1
$$

So:

$$
\frac{\partial L}{\partial b} = (a-y)f'(z)
$$

This already shows the pattern:

- output error term
- times activation derivative
- times input to the parameter


## 4. From one neuron to a multilayer network

Consider a 2-layer network:

$$
a^{[0]} = x
$$

Hidden layer:

$$
z^{[1]} = W^{[1]}a^{[0]} + b^{[1]}
$$

$$
a^{[1]} = f^{[1]}(z^{[1]})
$$

Output layer:

$$
z^{[2]} = W^{[2]}a^{[1]} + b^{[2]}
$$

$$
a^{[2]} = f^{[2]}(z^{[2]})
$$

Loss:

$$
L = \mathcal{L}(a^{[2]}, y)
$$

We want:

$$
\frac{\partial L}{\partial W^{[2]}},\;
\frac{\partial L}{\partial b^{[2]}},\;
\frac{\partial L}{\partial W^{[1]}},\;
\frac{\partial L}{\partial b^{[1]}}
$$


## 5. Error terms: the clean notation

A very useful quantity is:

$$
\delta^{[l]} = \frac{\partial L}{\partial z^{[l]}}
$$

This is the gradient of the loss with respect to the **pre-activation** at layer $l$.

Why is this useful? Because once you know $\delta^{[l]}$, gradients become simple:

$$
\frac{\partial L}{\partial W^{[l]}} = \delta^{[l]} (a^{[l-1]})^T
$$

$$
\frac{\partial L}{\partial b^{[l]}} = \delta^{[l]}
$$

for one example.

So backpropagation mainly means:

1. compute $\delta$ at the output layer
2. propagate it backward through earlier layers


## 6. Output layer gradient

For the last layer:

$$
\delta^{[2]} = \frac{\partial L}{\partial z^{[2]}}
$$

By chain rule:

$$
\delta^{[2]}
=
\frac{\partial L}{\partial a^{[2]}}
\odot
f'^{[2]}(z^{[2]})
$$

where $\odot$ is elementwise multiplication.

Then:

$$
\frac{\partial L}{\partial W^{[2]}}
=
\delta^{[2]}(a^{[1]})^T
$$

$$
\frac{\partial L}{\partial b^{[2]}}
=
\delta^{[2]}
$$


## 7. Hidden layer gradient

The hidden layer affects the loss only through the later layer. So:

$$
\delta^{[1]} =
\left((W^{[2]})^T \delta^{[2]}\right)\odot f'^{[1]}(z^{[1]})
$$

This is the essential backprop formula.

Interpretation:

- $\left((W^{[2]})^T \delta^{[2]}\right)$: how much the next layer says this hidden unit mattered
- multiply by $f'^{[1]}(z^{[1]})$: local sensitivity of this hidden unit

Then:

$$
\frac{\partial L}{\partial W^{[1]}}
=
\delta^{[1]}(a^{[0]})^T
$$

$$
\frac{\partial L}{\partial b^{[1]}}
=
\delta^{[1]}
$$


## 8. General formula for any layer

For any layer $l$:

Forward:

$$
z^{[l]} = W^{[l]}a^{[l-1]} + b^{[l]}
$$

$$
a^{[l]} = f^{[l]}(z^{[l]})
$$

Backward:

### Output layer

$$
\delta^{[L]} = \frac{\partial L}{\partial a^{[L]}} \odot f'^{[L]}(z^{[L]})
$$

### Hidden layers

$$
\delta^{[l]} = \left(W^{[l+1]T}\delta^{[l+1]}\right)\odot f'^{[l]}(z^{[l]})
$$

### Parameter gradients

$$
\frac{\partial L}{\partial W^{[l]}} = \delta^{[l]}(a^{[l-1]})^T
$$

$$
\frac{\partial L}{\partial b^{[l]}} = \delta^{[l]}
$$

This is the compact mathematical summary of backpropagation.


## 9. Worked numerical example

Let us do a complete small network by hand.

We will use:

- input dimension = 2
- hidden layer = 2 neurons
- output layer = 1 neuron
- sigmoid activation everywhere
- one training example
- mean squared error loss

### Given

Input:

$$
x = \begin{bmatrix} 1 \\ 2 \end{bmatrix}
$$

Target:

$$
y = 1
$$

#### Hidden layer parameters

$$
W^{[1]} =
\begin{bmatrix}
0.1 & 0.2 \\
0.3 & 0.4
\end{bmatrix}
$$

$$
b^{[1]} =
\begin{bmatrix}
0.1 \\
0.1
\end{bmatrix}
$$

#### Output layer parameters

$$
W^{[2]} =
\begin{bmatrix}
0.5 & 0.6
\end{bmatrix}
$$

$$
b^{[2]} = \begin{bmatrix} 0.1 \end{bmatrix}
$$

Sigmoid:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Derivative:

$$
\sigma'(z)=\sigma(z)(1-\sigma(z))
$$


### Step 1: forward pass

#### Hidden pre-activation

$$
z^{[1]} = W^{[1]}x + b^{[1]}
$$

$$
=
\begin{bmatrix}
0.1 & 0.2 \\
0.3 & 0.4
\end{bmatrix}
\begin{bmatrix}
1\\
2
\end{bmatrix}
+
\begin{bmatrix}
0.1\\
0.1
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
0.1(1)+0.2(2)\\
0.3(1)+0.4(2)
\end{bmatrix}
+
\begin{bmatrix}
0.1\\
0.1
\end{bmatrix}
=
\begin{bmatrix}
0.5\\
1.1
\end{bmatrix}
+
\begin{bmatrix}
0.1\\
0.1
\end{bmatrix}
=
\begin{bmatrix}
0.6\\
1.2
\end{bmatrix}
$$

#### Hidden activation

$$
a^{[1]} = \sigma(z^{[1]})
$$

$$
a^{[1]} \approx
\begin{bmatrix}
\sigma(0.6)\\
\sigma(1.2)
\end{bmatrix}
=
\begin{bmatrix}
0.6457\\
0.7685
\end{bmatrix}
$$

#### Output pre-activation

$$
z^{[2]} = W^{[2]}a^{[1]} + b^{[2]}
$$

$$
=
\begin{bmatrix}
0.5 & 0.6
\end{bmatrix}
\begin{bmatrix}
0.6457\\
0.7685
\end{bmatrix}
+ 0.1
$$

$$
= 0.5(0.6457)+0.6(0.7685)+0.1
$$

$$
=0.32285+0.4611+0.1 = 0.88395
$$

#### Output activation

$$
a^{[2]} = \sigma(0.88395) \approx 0.7076
$$

#### Loss

$$
L = \frac{1}{2}(a^{[2]}-y)^2
$$

$$
= \frac{1}{2}(0.7076-1)^2
\approx \frac{1}{2}( -0.2924)^2
\approx 0.0427
$$


### Step 2: backward pass at output layer

We compute:

$$
\delta^{[2]} = \frac{\partial L}{\partial z^{[2]}}
$$

Using chain rule:

$$
\delta^{[2]}
=
\frac{\partial L}{\partial a^{[2]}}
\cdot
\frac{\partial a^{[2]}}{\partial z^{[2]}}
$$

Now:

$$
\frac{\partial L}{\partial a^{[2]}} = a^{[2]}-y = 0.7076 - 1 = -0.2924
$$

$$
\frac{\partial a^{[2]}}{\partial z^{[2]}} = a^{[2]}(1-a^{[2]})
= 0.7076(1-0.7076)
\approx 0.2070
$$

So:

$$
\delta^{[2]} \approx -0.2924 \times 0.2070 = -0.0605
$$

#### Gradients for output layer

$$
\frac{\partial L}{\partial W^{[2]}} = \delta^{[2]}(a^{[1]})^T
$$

$$
= -0.0605
\begin{bmatrix}
0.6457 & 0.7685
\end{bmatrix}
$$

$$
\approx
\begin{bmatrix}
-0.0391 & -0.0465
\end{bmatrix}
$$

And:

$$
\frac{\partial L}{\partial b^{[2]}} = \delta^{[2]} \approx -0.0605
$$


### Step 3: backpropagate to hidden layer

$$
\delta^{[1]} = (W^{[2]T}\delta^{[2]}) \odot \sigma'(z^{[1]})
$$

First compute:

$$
W^{[2]T}\delta^{[2]}
=
\begin{bmatrix}
0.5\\
0.6
\end{bmatrix}
(-0.0605)
=
\begin{bmatrix}
-0.03025\\
-0.03630
\end{bmatrix}
$$

Now compute sigmoid derivatives at hidden layer:

$$
\sigma'(0.6)=0.6457(1-0.6457)\approx 0.2288
$$

$$
\sigma'(1.2)=0.7685(1-0.7685)\approx 0.1779
$$

So:

$$
\delta^{[1]}
=
\begin{bmatrix}
-0.03025\\
-0.03630
\end{bmatrix}
\odot
\begin{bmatrix}
0.2288\\
0.1779
\end{bmatrix}
\approx
\begin{bmatrix}
-0.00692\\
-0.00646
\end{bmatrix}
$$

#### Gradients for hidden layer weights

$$
\frac{\partial L}{\partial W^{[1]}} = \delta^{[1]} x^T
$$

$$
=
\begin{bmatrix}
-0.00692\\
-0.00646
\end{bmatrix}
\begin{bmatrix}
1 & 2
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
-0.00692 & -0.01384\\
-0.00646 & -0.01292
\end{bmatrix}
$$

#### Gradients for hidden biases

$$
\frac{\partial L}{\partial b^{[1]}} =
\begin{bmatrix}
-0.00692\\
-0.00646
\end{bmatrix}
$$


## 10. Parameter update

Suppose learning rate:

$$
\eta = 0.1
$$

Then:

$$
W^{[2]} \leftarrow W^{[2]} - 0.1 \frac{\partial L}{\partial W^{[2]}}
$$

$$
=
\begin{bmatrix}
0.5 & 0.6
\end{bmatrix}
-
0.1
\begin{bmatrix}
-0.0391 & -0.0465
\end{bmatrix}
$$

$$
=
\begin{bmatrix}
0.50391 & 0.60465
\end{bmatrix}
$$

Similarly:

$$
b^{[2]} \leftarrow 0.1 - 0.1(-0.0605)=0.10605
$$

And hidden layer parameters update the same way.

Because gradients were negative here, the weights slightly increase.


## 11. What is really happening intuitively?

Backpropagation answers:

- the output was too low or too high
- which output weights caused that?
- which hidden neurons contributed to that wrong output?
- which earlier weights caused those hidden neurons to activate the way they did?

So error is not literally “sent backward” as a physical thing.  
What is propagated backward is **credit/blame information** in the form of gradients.


## 12. Why transpose appears

For hidden layers:

$$
\delta^{[l]} = (W^{[l+1]T}\delta^{[l+1]}) \odot f'(z^{[l]})
$$

Why transpose?

Because $W^{[l+1]}$ maps activations from layer $l$ to layer $l+1$. During backprop, we want to move influence in the reverse direction: from layer $l+1$ back to $l$. That is why the transpose appears.

Forward:

$$
a^{[l]} \to z^{[l+1]}
$$

Backward:

$$
\delta^{[l+1]} \to \delta^{[l]}
$$

The transpose reverses the linear map.


## 13. Why activations matter in weight gradients

For any weight connecting neuron $j$ in previous layer to neuron $i$ in current layer:

$$
z_i^{[l]} = \sum_j W_{ij}^{[l]} a_j^{[l-1]} + b_i^{[l]}
$$

So:

$$
\frac{\partial z_i^{[l]}}{\partial W_{ij}^{[l]}} = a_j^{[l-1]}
$$

Hence:

$$
\frac{\partial L}{\partial W_{ij}^{[l]}}
=
\delta_i^{[l]} a_j^{[l-1]}
$$

This is a beautiful formula:

> gradient for a weight = error at destination neuron × activation at source neuron

Very important intuition.


## 14. Batch version

For one example, gradients are as above.

For a mini-batch of $m$ examples, we usually average gradients:

$$
\frac{\partial J}{\partial W^{[l]}}
=
\frac{1}{m}\sum_{k=1}^{m}
\frac{\partial L^{(k)}}{\partial W^{[l]}}
$$

In matrix form, if columns are examples:

$$
dW^{[l]} = \frac{1}{m} \delta^{[l]} (A^{[l-1]})^T
$$

$$
db^{[l]} = \frac{1}{m}\sum \delta^{[l]}
$$

This is how practical training is done.


## 15. Special simplification: sigmoid/softmax with cross-entropy

This is extremely important.

For sigmoid output with binary cross-entropy, the output gradient simplifies to:

$$
\delta^{[L]} = a^{[L]} - y
$$

For softmax with cross-entropy, similarly:

$$
\delta^{[L]} = \hat{y} - y
$$

This is one reason these combinations are so popular. The derivative becomes cleaner and numerically better behaved.


## 16. Common conceptual mistakes

### Mistake 1: thinking backprop is separate from chain rule

It is not. Backpropagation is just an efficient organization of repeated chain rule calculations.

### Mistake 2: thinking gradients are computed independently for each weight

Not from scratch. Backprop reuses intermediate gradients, which makes it efficient.

### Mistake 3: confusing activations and pre-activations

- $z$: before nonlinearity
- $a$: after nonlinearity

The derivative of the activation function is with respect to $z$.

### Mistake 4: thinking the error signal is the same at all layers

No. Each layer’s error term is transformed by:

- downstream weights
- local activation derivative


## 17. Why backpropagation is efficient

Imagine computing $\frac{\partial L}{\partial w}$ separately for every weight by expanding every path from that weight to the output. That would repeat huge amounts of work.

Backprop avoids this by computing downstream gradients once and reusing them.

That is dynamic programming over the computation graph.


## 18. Minimal NumPy code

Here is a clean implementation for the exact 2-layer example above.


In [ ]:
import numpy as np

# ----------------------------
# Helper functions
# ----------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative_from_activation(a):
    # If a = sigmoid(z), then sigmoid'(z) = a * (1 - a)
    return a * (1 - a)

# ----------------------------
# Data: one example
# ----------------------------
x = np.array([[1.0],
              [2.0]])   # shape (2,1)

y = np.array([[1.0]])   # shape (1,1)

# ----------------------------
# Parameters
# ----------------------------
W1 = np.array([[0.1, 0.2],
               [0.3, 0.4]])   # shape (2,2)

b1 = np.array([[0.1],
               [0.1]])        # shape (2,1)

W2 = np.array([[0.5, 0.6]])   # shape (1,2)
b2 = np.array([[0.1]])        # shape (1,1)

learning_rate = 0.1

# ----------------------------
# Forward pass
# ----------------------------
z1 = W1 @ x + b1
a1 = sigmoid(z1)

z2 = W2 @ a1 + b2
a2 = sigmoid(z2)

loss = 0.5 * np.sum((a2 - y) ** 2)

print("Forward pass:")
print("z1 =\n", z1)
print("a1 =\n", a1)
print("z2 =\n", z2)
print("a2 =\n", a2)
print("loss =", loss)

# ----------------------------
# Backward pass
# ----------------------------
# Output layer
dL_da2 = a2 - y
da2_dz2 = sigmoid_derivative_from_activation(a2)
delta2 = dL_da2 * da2_dz2   # dL/dz2

dW2 = delta2 @ a1.T
db2 = delta2

# Hidden layer
delta1 = (W2.T @ delta2) * sigmoid_derivative_from_activation(a1)

dW1 = delta1 @ x.T
db1 = delta1

print("\nBackward pass:")
print("delta2 =\n", delta2)
print("dW2 =\n", dW2)
print("db2 =\n", db2)
print("delta1 =\n", delta1)
print("dW1 =\n", dW1)
print("db1 =\n", db1)

# ----------------------------
# Parameter update
# ----------------------------
W2_new = W2 - learning_rate * dW2
b2_new = b2 - learning_rate * db2
W1_new = W1 - learning_rate * dW1
b1_new = b1 - learning_rate * db1

print("\nUpdated parameters:")
print("W2_new =\n", W2_new)
print("b2_new =\n", b2_new)
print("W1_new =\n", W1_new)
print("b1_new =\n", b1_new)

## 19. Slightly more general code for one hidden layer

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_prime(a):
    return a * (1 - a)

class TinyMLP:
    def __init__(self, input_dim, hidden_dim, output_dim, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(scale=0.1, size=(hidden_dim, input_dim))
        self.b1 = np.zeros((hidden_dim, 1))
        self.W2 = rng.normal(scale=0.1, size=(output_dim, hidden_dim))
        self.b2 = np.zeros((output_dim, 1))

    def forward(self, x):
        self.x = x
        self.z1 = self.W1 @ x + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = self.W2 @ self.a1 + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    def loss(self, y_hat, y):
        return 0.5 * np.mean((y_hat - y) ** 2)

    def backward(self, y):
        m = y.shape[1]  # batch size

        delta2 = (self.a2 - y) * sigmoid_prime(self.a2)   # shape (output_dim, m)
        dW2 = (delta2 @ self.a1.T) / m
        db2 = np.sum(delta2, axis=1, keepdims=True) / m

        delta1 = (self.W2.T @ delta2) * sigmoid_prime(self.a1)
        dW1 = (delta1 @ self.x.T) / m
        db1 = np.sum(delta1, axis=1, keepdims=True) / m

        return dW1, db1, dW2, db2

    def step(self, grads, lr=0.1):
        dW1, db1, dW2, db2 = grads
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2


# Example usage
X = np.array([[0.0, 0.0, 1.0, 1.0],
              [0.0, 1.0, 0.0, 1.0]])   # shape (2,4)

Y = np.array([[0.0, 1.0, 1.0, 0.0]])   # XOR-like targets

model = TinyMLP(input_dim=2, hidden_dim=3, output_dim=1, seed=42)

for epoch in range(5000):
    y_hat = model.forward(X)
    loss = model.loss(y_hat, Y)
    grads = model.backward(Y)
    model.step(grads, lr=1.0)

    if epoch % 500 == 0:
        print(f"epoch={epoch}, loss={loss:.6f}")

print("\nPredictions:")
print(model.forward(X))

## 20. Backpropagation in computational graph language

Suppose we break computation into tiny operations:

- multiply
- add
- activation
- loss

Each node knows how to compute:

- its forward value
- local derivative with respect to its inputs

Then backprop is:

$$
\text{upstream gradient} \times \text{local derivative}
$$

This is exactly how autodiff systems like PyTorch work internally.


## 21. Relation to PyTorch autograd

In PyTorch, you usually do not write backprop manually. You do:

```python
loss.backward()
```

PyTorch has already recorded the computation graph during the forward pass. Then it applies the same chain rule backward through the graph.

So:

- **manual backprop** = learning the math
- **autograd** = software automation of the same math


## 22. Why vanishing gradients happen

Backprop multiplies many derivatives together. If those derivatives are often less than 1, then over many layers the gradient can shrink toward zero.

Example:

$$
0.2 \times 0.3 \times 0.4 \times 0.2 \times 0.1
$$

becomes tiny very quickly.

This is why sigmoid/tanh in deep networks can struggle, especially in older architectures.

ReLU helps because its derivative is often 1 in the active region.


## 23. Compact summary to remember

Here is the shortest “mental template” for backprop:

### Forward

$$
z^{[l]} = W^{[l]}a^{[l-1]} + b^{[l]}
$$

$$
a^{[l]} = f(z^{[l]})
$$

### Backward

Output:

$$
\delta^{[L]} = \frac{\partial L}{\partial a^{[L]}} \odot f'(z^{[L]})
$$

Hidden:

$$
\delta^{[l]} = (W^{[l+1]T}\delta^{[l+1]}) \odot f'(z^{[l]})
$$

Gradients:

$$
dW^{[l]} = \delta^{[l]}(a^{[l-1]})^T
$$

$$
db^{[l]} = \delta^{[l]}
$$

Update:

$$
W^{[l]} \leftarrow W^{[l]} - \eta dW^{[l]}
$$

$$
b^{[l]} \leftarrow b^{[l]} - \eta db^{[l]}
$$


## 24. The deepest intuition in one sentence

Backpropagation tells every parameter:

> “How much did you contribute to the final mistake, through all the downstream computations?”

That contribution is measured by the gradient.


## 25. Optional quick verification cell

This cell recomputes the numerical example and prints rounded values so you can compare them directly with the worked derivation above.


In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.array([[1.0], [2.0]])
y = np.array([[1.0]])

W1 = np.array([[0.1, 0.2],
               [0.3, 0.4]])
b1 = np.array([[0.1], [0.1]])

W2 = np.array([[0.5, 0.6]])
b2 = np.array([[0.1]])

z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
a2 = sigmoid(z2)
L = 0.5 * (a2 - y) ** 2

delta2 = (a2 - y) * a2 * (1 - a2)
dW2 = delta2 @ a1.T
db2 = delta2

delta1 = (W2.T @ delta2) * a1 * (1 - a1)
dW1 = delta1 @ x.T
db1 = delta1

print("z1 =", np.round(z1, 5).ravel())
print("a1 =", np.round(a1, 5).ravel())
print("z2 =", np.round(z2, 5).ravel())
print("a2 =", np.round(a2, 5).ravel())
print("L  =", np.round(L, 5).ravel())
print("delta2 =", np.round(delta2, 5).ravel())
print("dW2 =\n", np.round(dW2, 5))
print("db2 =", np.round(db2, 5).ravel())
print("delta1 =", np.round(delta1, 5).ravel())
print("dW1 =\n", np.round(dW1, 5))
print("db1 =", np.round(db1, 5).ravel())